# 2.3. Подготовка и интеграция данных

## Цель этапа

На данном этапе выполняется подготовка данных, полученных в рамках этапа 2.2, к последующему статистическому анализу.

Основные задачи:

1. Инвентаризация исходных и ранее подготовленных данных.
2. Профилирование наборов данных: структура, типы, пропуски, дубликаты и диапазоны значений.
3. Очистка и преобразование данных с сохранением исходных файлов.
4. Подготовка текстовых данных Steam Reviews, YouTube и Habr.
5. Формирование текстовых признаков с использованием TF-IDF.
6. Разбор и преобразование Wikidata XML.
7. Сопоставление игр из разных источников.
8. Формирование аналитических таблиц.
9. Сохранение подготовленных данных в Parquet и аналитического хранилища DuckDB.
10. Проверка качества и воспроизводимости результатов.

Исходные файлы в каталоге `raw/` не изменяются. Все результаты обработки сохраняются отдельно в `processed/` и `warehouse/`.

При обработке учитываются особенности каждого источника. Идентификаторы разных платформ не считаются взаимозаменяемыми, а совпадения названий игр требуют проверки.

Работа выполняется последовательно: сначала изучается фактическая структура данных, затем принимаются решения об их очистке и преобразовании.

## 1. Инвентаризация исходных и ранее подготовленных данных.

In [1]:
from pathlib import Path
from datetime import datetime
import os
import json
import sys

import pandas as pd
from IPython.display import display

# Определяем корень проекта независимо от того,
# запущен ноутбук из корня или из каталога notebooks.

current_path = Path.cwd()

ROOT = None

for path in [current_path, *current_path.parents]:
    if (path / "notebooks").is_dir() and (path / "README.md").exists():
        ROOT = path
        break

if ROOT is None:
    raise FileNotFoundError(
        "Не удалось определить корень проекта. "
        "Запусти ноутбук внутри gaming-data-analysis."
    )

RAW = ROOT / "raw"
PROCESSED = ROOT / "processed"
WAREHOUSE = ROOT / "warehouse"
DOCS = ROOT / "docs"
QUALITY = DOCS / "data_quality"

if not RAW.exists():
    raise FileNotFoundError(
        f"Каталог исходных данных не найден: {RAW}"
    )

QUALITY.mkdir(parents=True, exist_ok=True)

print("Корень проекта:", ROOT)
print("Исходные данные:", RAW)
print("Подготовленные данные:", PROCESSED)
print("Аналитическое хранилище:", WAREHOUSE)
print("Отчёты о качестве:", QUALITY)

Корень проекта: C:\Users\245108\Desktop\gaming-data-analysis
Исходные данные: C:\Users\245108\Desktop\gaming-data-analysis\raw
Подготовленные данные: C:\Users\245108\Desktop\gaming-data-analysis\processed
Аналитическое хранилище: C:\Users\245108\Desktop\gaming-data-analysis\warehouse
Отчёты о качестве: C:\Users\245108\Desktop\gaming-data-analysis\docs\data_quality


In [2]:
data_directories = {
    "raw": RAW,
    "processed": PROCESSED,
    "warehouse": WAREHOUSE,
}

inventory_records = []

for layer, directory in data_directories.items():

    if not directory.exists():
        print(f"Каталог отсутствует: {directory}")
        continue

    for path in directory.rglob("*"):

        if not path.is_file():
            continue

        file_size = path.stat().st_size

        relative_path = path.relative_to(ROOT)

        parts = relative_path.parts

        inventory_records.append({
            "layer": layer,
            "source_group": parts[1] if len(parts) > 1 else None,
            "dataset": str(Path(*parts[1:-1])) if len(parts) > 2 else None,
            "filename": path.name,
            "extension": path.suffix.lower(),
            "relative_path": str(relative_path),
            "size_bytes": file_size,
            "size_mb": round(file_size / 1024**2, 2),
            "modified_at": datetime.fromtimestamp(
                path.stat().st_mtime
            ).isoformat(timespec="seconds"),
        })

inventory = pd.DataFrame(inventory_records)

if inventory.empty:
    raise ValueError("В каталогах проекта не найдено файлов.")

inventory = inventory.sort_values(
    ["layer", "size_bytes"],
    ascending=[True, False]
).reset_index(drop=True)

display(inventory)

print(f"Всего файлов: {len(inventory)}")
print(
    f"Общий размер: "
    f"{inventory['size_bytes'].sum() / 1024**3:.2f} GB"
)

Каталог отсутствует: C:\Users\245108\Desktop\gaming-data-analysis\warehouse


,layer,source_group,dataset,filename,extension,relative_path,size_bytes,size_mb,modified_at
0,processed,games,games,rawg_games.parquet,.parquet,processed\games\rawg_games.parquet,826822431,788.52,2026-10-19T12:42:09
1,processed,games,games,steam_games_json.parquet,.parquet,processed\games\steam_games_json.parquet,247428327,235.97,2026-10-19T12:53:31
2,processed,games,games,steam_games.parquet,.parquet,processed\games\steam_games.parquet,245950525,234.56,2026-10-19T12:40:03
3,processed,reviews,reviews,steam_2025_reviews.parquet,.parquet,processed\reviews\steam_2025_reviews.parquet,233814272,222.98,2026-10-19T12:55:31
4,processed,steam_2025,steam_2025,applications.parquet,.parquet,processed\steam_2025\applications.parquet,30513875,29.10,2026-10-19T12:52:36
...,...,...,...,...,...,...,...,...,...
62,raw,epic,epic,epic_games_editor.csv,.csv,raw\epic\epic_games_editor.csv,5309,0.01,2026-10-05T12:08:10
63,raw,epic,epic,epic-games-apps.csv,.csv,raw\epic\epic-games-apps.csv,4202,0.00,2026-10-05T12:08:10
64,raw,steam,steam\games_2025\steam_dataset_2025_csv,MANIFEST.json,.json,raw\steam\games_2025\steam_dataset_2025_csv\MA...,2878,0.00,2026-10-05T12:10:46
65,raw,steam,steam\games_2025\steam_dataset_2025_csv,genres.csv,.csv,raw\steam\games_2025\steam_dataset_2025_csv\ge...,2447,0.00,2026-10-05T12:10:47


Всего файлов: 67
Общий размер: 16.29 GB


In [3]:
# Количество файлов и общий объём по слоям хранения

inventory_by_layer = (
    inventory
    .groupby("layer")
    .agg(
        files_count=("filename", "count"),
        total_size_mb=("size_mb", "sum"),
    )
    .reset_index()
)

inventory_by_layer["total_size_gb"] = (
    inventory_by_layer["total_size_mb"] / 1024
).round(2)

display(inventory_by_layer)


# Количество файлов и общий объём по расширениям

inventory_by_extension = (
    inventory
    .groupby("extension")
    .agg(
        files_count=("filename", "count"),
        total_size_mb=("size_mb", "sum"),
    )
    .reset_index()
    .sort_values("total_size_mb", ascending=False)
)

display(inventory_by_extension)


# Десять крупнейших файлов

display(
    inventory.nlargest(10, "size_bytes")[
        [
            "layer",
            "relative_path",
            "size_mb",
            "extension",
        ]
    ]
)

,layer,files_count,total_size_mb,total_size_gb
0,processed,20,1523.12,1.49
1,raw,47,15159.07,14.80


,extension,files_count,total_size_mb
0,.csv,37,9101.78
2,.jsonl,1,4101.47
3,.parquet,21,2487.92
1,.json,6,990.31
5,.xml,1,0.70
4,.py,1,0.01


,layer,relative_path,size_mb,extension
20,raw,raw\rawg\jsonl\videogames_data.jsonl,4101.47,.jsonl
21,raw,raw\steam\reviews\dataset.csv,2060.77,.csv
22,raw,raw\steam\recommendations\recommendations.csv,1929.50,.csv
23,raw,raw\rawg\videogames_data.csv,1907.80,.csv
24,raw,raw\steam\fronkongames\games.json,973.38,.json
25,raw,raw\rawg\videogames_data.parquet,964.80,.parquet
0,processed,processed\games\rawg_games.parquet,788.52,.parquet
26,raw,raw\steam\games_2025\steam_dataset_2025_csv\re...,638.78,.csv
27,raw,raw\steam\games_artemiloff\games_march2025_ful...,449.65,.csv
28,raw,raw\steam\games_artemiloff\games_march2025_cle...,446.93,.csv


In [4]:
inventory_path = QUALITY / "file_inventory.csv"

inventory.to_csv(
    inventory_path,
    index=False,
    encoding="utf-8-sig",
)

print("Инвентаризация сохранена:", inventory_path)
print("Размер отчёта:", round(inventory_path.stat().st_size / 1024, 2), "KB")

Инвентаризация сохранена: C:\Users\245108\Desktop\gaming-data-analysis\docs\data_quality\file_inventory.csv
Размер отчёта: 9.22 KB


### Уточнение состава файлов перед профилированием

In [5]:
# Полный перечень исходных файлов

raw_files = (
    inventory[inventory["layer"] == "raw"]
    .sort_values(["source_group", "relative_path"])
)

display(
    raw_files[
        [
            "source_group",
            "relative_path",
            "extension",
            "size_mb",
        ]
    ].to_string(index=False)
)

# Полный перечень ранее подготовленных файлов

processed_files = (
    inventory[inventory["layer"] == "processed"]
    .sort_values("relative_path")
)

display(
    processed_files[
        [
            "relative_path",
            "extension",
            "size_mb",
            "modified_at",
        ]
    ].to_string(index=False)
)

'source_group                                                          relative_path extension  size_mb\n        epic                              raw\\epic\\epic games store-video games.csv      .csv     0.25\n        epic                                           raw\\epic\\epic-games-apps.csv      .csv     0.00\n        epic                                           raw\\epic\\epic-games-demo.csv      .csv     0.01\n        epic                                         raw\\epic\\epic_games_editor.csv      .csv     0.01\n       games                    raw\\games\\ultimate_games\\Ultimate_Games_Dataset.csv      .csv    27.54\n        habr                                                 raw\\habr\\articles.json     .json     0.03\n        rawg                                   raw\\rawg\\jsonl\\videogames_data.jsonl    .jsonl  4101.47\n        rawg                                           raw\\rawg\\videogames_data.csv      .csv  1907.80\n        rawg                                 

'                                      relative_path extension  size_mb         modified_at\n                 processed\\games\\epic_games.parquet  .parquet     0.09 2026-10-19T12:46:19\n                 processed\\games\\rawg_games.parquet  .parquet   788.52 2026-10-19T12:42:09\n                processed\\games\\steam_games.parquet  .parquet   234.56 2026-10-19T12:40:03\n           processed\\games\\steam_games_json.parquet  .parquet   235.97 2026-10-19T12:53:31\n    processed\\players\\steam_monthly_players.parquet  .parquet     5.90 2026-10-19T12:46:20\n       processed\\reviews\\steam_2025_reviews.parquet  .parquet   222.98 2026-10-19T12:55:31\nprocessed\\steam_2025\\application_categories.parquet  .parquet     1.23 2026-10-19T12:52:36\nprocessed\\steam_2025\\application_developers.parquet  .parquet     0.84 2026-10-19T12:52:36\n    processed\\steam_2025\\application_genres.parquet  .parquet     0.56 2026-10-19T12:52:37\n processed\\steam_2025\\application_platforms.parquet  .parqu

## 2. Профилирование и оценка качества исходных данных

На данном этапе проводится первичное исследование структуры исходных наборов данных. Цель — определить состав полей, форматы хранения, предполагаемые типы данных и особенности организации информации.

Для больших файлов используется ограниченное чтение или потоковый просмотр. Полная загрузка крупных CSV, JSON и JSONL в оперативную память на этапе первичного профилирования не выполняется.

По результатам профилирования будут определены:
- структура и схема каждого источника;
- потенциальные ключи для идентификации объектов и связи между наборами;
- дублирующие и альтернативные представления данных;
- особенности, требующие дополнительной очистки;
- источники, подходящие для дальнейшего аналитического исследования.

In [15]:

from pathlib import Path
import json
import pandas as pd

from IPython.display import display

# Папки RAW и PROCESSED должны быть определены в предыдущих ячейках.

profiles = []

def inspect_source_file(path: Path):
    relative_path = str(path.relative_to(ROOT))
    extension = path.suffix.lower()

    result = {
        "relative_path": relative_path,
        "extension": extension,
        "columns": None,
        "sample_rows": None,
        "structure": None,
        "status": "OK",
    }

    try:
        if extension == ".csv":
            # Читаем только заголовок и первые 5 строк
            sample = pd.read_csv(
                path,
                nrows=5,
                encoding="utf-8",
                low_memory=False
            )

            result["columns"] = list(sample.columns)
            result["sample_rows"] = len(sample)
            result["structure"] = "Tabular CSV"

        elif extension == ".parquet":
            import pyarrow.parquet as pq

            parquet_file = pq.ParquetFile(path)

            result["columns"] = parquet_file.schema.names
            result["sample_rows"] = min(
                parquet_file.metadata.num_rows, 5
            )
            result["structure"] = (
                f"Parquet; rows={parquet_file.metadata.num_rows}; "
                f"row_groups={parquet_file.num_row_groups}"
            )

        elif extension == ".jsonl":
            # Считываем только первые 3 записи
            with open(path, "r", encoding="utf-8") as file:
                lines = [file.readline() for _ in range(3)]

            records = [
                json.loads(line)
                for line in lines
                if line.strip()
            ]

            if records:
                result["columns"] = list(records[0].keys())

            result["sample_rows"] = len(records)
            result["structure"] = "JSON Lines"

        elif extension == ".json":
            # Не загружаем весь JSON:
            # смотрим только начало файла.
            with open(path, "r", encoding="utf-8") as file:
                preview = file.read(4096).lstrip()

            if preview.startswith("{"):
                structure = "JSON object"
            elif preview.startswith("["):
                structure = "JSON array"
            else:
                structure = "Unknown JSON structure"

            result["structure"] = structure
            result["sample_rows"] = None

        elif extension == ".xml":
            import xml.etree.ElementTree as ET

            # Получаем корневой элемент, не загружая весь XML.
            for event, element in ET.iterparse(
                path, events=("start",)
            ):
                result["structure"] = (
                    f"XML; root={element.tag}"
                )
                break

        elif extension == ".py":
            result["structure"] = "Python script"

        else:
            result["structure"] = "Unsupported format"

    except Exception as error:
        result["status"] = (
            f"ERROR: {type(error).__name__}: {error}"
        )

    return result


# Профилируем все исходные файлы
for path in sorted(RAW.rglob("*")):
    if not path.is_file():
        continue

    if path.suffix.lower() not in {
        ".csv", ".parquet", ".json",
        ".jsonl", ".xml", ".py"
    }:
        continue

    profiles.append(inspect_source_file(path))


source_profiles = pd.DataFrame(profiles)

print(f"Профилировано файлов: {len(source_profiles)}")
print("\nСтатусы выполнения:")
display(source_profiles["status"].value_counts())

print("\nСтруктура исходных файлов:")
display(
    source_profiles[
        [
            "relative_path",
            "extension",
            "columns",
            "sample_rows",
            "structure",
            "status"
        ]
    ]#.to_string(index=False)
)


Профилировано файлов: 47

Статусы выполнения:


status
OK    47
Name: count, dtype: int64


Структура исходных файлов:


,relative_path,extension,columns,sample_rows,structure,status
0,raw\epic\epic games store-video games.csv,.csv,"[game_link, name, price of game, developer of ...",5.0,Tabular CSV,OK
1,raw\epic\epic-games-apps.csv,.csv,"[links, name, type, genres, features, price, d...",5.0,Tabular CSV,OK
2,raw\epic\epic-games-demo.csv,.csv,"[links, name, type, genres, features, Develope...",5.0,Tabular CSV,OK
3,raw\epic\epic_games_editor.csv,.csv,"[links, name, type, genres, features, Develope...",5.0,Tabular CSV,OK
4,raw\games\ultimate_games\Ultimate_Games_Datase...,.csv,"[serial_no, game_id, title, release_date, rele...",5.0,Tabular CSV,OK
5,raw\habr\articles.json,.json,None,NaN,JSON array,OK
6,raw\rawg\jsonl\videogames_data.jsonl,.jsonl,"[id, slug, name, released, background_image, r...",3.0,JSON Lines,OK
7,raw\rawg\videogames_data.csv,.csv,"[id, slug, name, released, background_image, r...",5.0,Tabular CSV,OK
8,raw\rawg\videogames_data.parquet,.parquet,"[id, slug, name, released, background_image, r...",5.0,Parquet; rows=899585; row_groups=1,OK
9,raw\steam\fronkongames\games.csv,.csv,"[AppID, Name, Release date, Estimated owners, ...",5.0,Tabular CSV,OK


### 2.1.1. Профилирование каталога игр Steam -- Steam FronkonGames Dataset (JSON)
В качестве первого источника рассматривается набор FronkonGames Steam Games Dataset. Он содержит сведения об играх платформы Steam, включая информацию о выпуске, стоимости, пользовательских оценках, разработчиках, издателях и игровых характеристиках.

На данном этапе изучаются структура набора данных, типы столбцов и примеры записей. Очистка и преобразование данных пока не выполняются.

Для анализа используется исходный JSON-файл FronkonGames. В отличие от CSV-версии, JSON сохраняет корректную структуру записей и соответствие идентификаторов игр их атрибутам.

Поскольку файл имеет большой объём, его первичное изучение выполняется потоковым чтением. На данном этапе проверяется структура JSON, состав полей и примеры записей.

In [16]:
import ijson
import pandas as pd

steam_json_path = RAW / "steam" / "fronkongames" / "games.json"

steam_records = []

with open(steam_json_path, "rb") as file:
    for app_id, game in ijson.kvitems(file, ""):
        steam_records.append({
            "AppID": app_id,
            **game
        })

        if len(steam_records) == 5:
            break

steam_sample = pd.DataFrame(steam_records)

print("Размерность выборки:", steam_sample.shape)
print("\nСтолбцы и типы данных:")
display(steam_sample.dtypes.to_frame("dtype"))

print("\nПервые 5 записей:")
display(steam_sample)

Размерность выборки: (5, 43)

Столбцы и типы данных:


,dtype
AppID,object
name,object
release_date,object
required_age,int64
price,object
dlc_count,int64
detailed_description,object
about_the_game,object
short_description,object
reviews,object



Первые 5 записей:


,AppID,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.0,0,,,,,...,0,0,0 - 0,0,0,0,0,0,0,[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,...,252,3,0 - 20000,8,0,8,0,65,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,...,21,3,0 - 20000,0,0,0,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,...,0,0,0 - 20000,0,0,0,0,0,1,[]
4,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,...,0,0,0 - 20000,0,0,0,0,0,0,[]


### 2.1.2. Полное профилирование Steam FronkonGames

Выполняется потоковый анализ всего JSON-файла. Определяются количество записей, пропуски по атрибутам и уникальность идентификаторов приложений. Это позволяет оценить полноту и пригодность данных для последующей обработки.

In [17]:
import ijson
from collections import Counter

field_counts = Counter()
app_ids = set()

total_rows = 0
missing_app_id = 0
duplicate_app_id = 0

with open(steam_json_path, "rb") as file:
    for app_id, game in ijson.kvitems(file, ""):
        total_rows += 1

        if not app_id:
            missing_app_id += 1
        elif app_id in app_ids:
            duplicate_app_id += 1
        else:
            app_ids.add(app_id)

        for column in steam_sample.columns[1:]:
            if column not in game or game[column] is None or game[column] == "":
                field_counts[column] += 1

steam_missing = pd.DataFrame({
    "missing_count": {
        column: field_counts[column]
        for column in steam_sample.columns
    }
})

steam_missing.loc["AppID", "missing_count"] = missing_app_id
steam_missing["missing_percent"] = (
    steam_missing["missing_count"] / total_rows * 100
).round(2)

print("Количество записей:", total_rows)
print("Количество уникальных AppID:", len(app_ids))
print("Количество повторяющихся AppID:", duplicate_app_id)

display(
    steam_missing.sort_values(
        "missing_percent", ascending=False
    )
)

Количество записей: 150466
Количество уникальных AppID: 150466
Количество повторяющихся AppID: 0


,missing_count,missing_percent
score_rank,150339,99.92
metacritic_url,146181,97.15
reviews,137693,91.51
notes,121784,80.94
website,92956,61.78
support_url,86325,57.37
support_email,25297,16.81
about_the_game,8543,5.68
detailed_description,8511,5.66
short_description,8334,5.54


### 2.1.3. Анализ вложенных полей

Отдельно исследуются поля, содержащие списки, словари и текстовые значения. Проверяется структура данных и распространённость пустых коллекций. Это необходимо для выбора способов преобразования данных в табличный формат.

In [18]:
from collections import Counter

nested_fields = [
    "reviews", "supported_languages", "full_audio_languages",
    "packages", "developers", "publishers", "categories",
    "genres", "screenshots", "movies", "tags"
]

field_types = {field: Counter() for field in nested_fields}
empty_values = Counter()

with open(steam_json_path, "rb") as file:
    for _, game in ijson.kvitems(file, ""):
        for field in nested_fields:
            value = game.get(field)

            field_types[field][type(value).__name__] += 1

            if value == [] or value == {} or value == "":
                empty_values[field] += 1

nested_profile = pd.DataFrame({
    "value_types": {
        field: dict(field_types[field])
        for field in nested_fields
    },
    "empty_count": {
        field: empty_values[field]
        for field in nested_fields
    }
})

nested_profile["empty_percent"] = (
    nested_profile["empty_count"] / total_rows * 100
).round(2)

display(nested_profile)

,value_types,empty_count,empty_percent
reviews,{'str': 150466},137693,91.51
supported_languages,{'list': 150466},8402,5.58
full_audio_languages,{'list': 150466},90353,60.05
packages,{'list': 150466},28996,19.27
developers,{'list': 150466},8456,5.62
publishers,{'list': 150466},8856,5.89
categories,{'list': 150466},8981,5.97
genres,{'list': 150466},8437,5.61
screenshots,{'list': 150466},6039,4.01
movies,{'list': 150466},150466,100.00


### 2.2.1. Профилирование качества данных RAWG

На данном этапе проводится оценка полноты и уникальности записей RAWG Games Dataset. Проверяются количество записей, уникальность идентификаторов и распределение пропущенных значений по атрибутам.

Особое внимание уделяется полям, характеризующим рейтинги, жанры, платформы, разработчиков и издателей, поскольку они могут использоваться при дальнейшем анализе и интеграции игровых данных.

In [26]:
import pyarrow.parquet as pq

rawg_path = RAW / "rawg" / "videogames_data.parquet"
rawg_file = pq.ParquetFile(rawg_path)

print("Количество записей:", rawg_file.metadata.num_rows)
print("Количество столбцов:", len(rawg_file.schema.names))

print("\nСтолбцы и типы данных:")
display(pd.DataFrame({
    "column": rawg_file.schema_arrow.names,
    "dtype": [str(t) for t in rawg_file.schema_arrow.types]
}))

print("\nПервые 3 записи:")
rawg_sample = next(rawg_file.iter_batches(batch_size=3)).to_pandas()
display(rawg_sample)


Количество записей: 899585
Количество столбцов: 54

Столбцы и типы данных:


,column,dtype
0,id,int64
1,slug,string
2,name,string
3,released,string
4,background_image,string
5,rating,double
6,rating_top,double
7,ratings_count,double
8,reviews_text_count,double
9,added,double



Первые 3 записи:


,id,slug,name,released,background_image,rating,rating_top,ratings_count,reviews_text_count,added,...,screenshots_count,parent_platforms,game_series_count,description,description_raw,parents_count,reddit_name,reddit_description,reddit_logo,tba
0,3498,grand-theft-auto-v,Grand Theft Auto V,2013-09-17,https://media.rawg.io/media/games/20a/20aa03a1...,4.47,5.0,7409.0,135.0,22635.0,...,55.0,PC|PlayStation|Xbox,12.0,"Rockstar Games went bigger, since their previo...","Rockstar Games went bigger, since their previo...",NaN,nan,nan,nan,NaN
1,3328,the-witcher-3-wild-hunt,The Witcher 3: Wild Hunt,2015-05-18,https://media.rawg.io/media/games/618/618c2031...,4.64,5.0,7234.0,120.0,22285.0,...,107.0,PC|PlayStation|Xbox|Apple Macintosh|Nintendo,9.0,"The third game in a series, it holds nothing b...","The third game in a series, it holds nothing b...",NaN,nan,nan,nan,NaN
2,4200,portal-2,Portal 2,2011-04-18,https://media.rawg.io/media/games/2ba/2bac0e87...,4.58,5.0,6118.0,57.0,20926.0,...,24.0,PC|PlayStation|Xbox|Apple Macintosh|Linux,3.0,Portal 2 is a first-person puzzle game develop...,Portal 2 is a first-person puzzle game develop...,NaN,nan,nan,nan,NaN


### 2.2.2 Профилирование пропусков

In [23]:
from collections import Counter
import pyarrow.parquet as pq

rawg_file = pq.ParquetFile(rawg_path)

missing_counts = Counter()
rawg_ids = set()
total_rows = 0

for batch in rawg_file.iter_batches(batch_size=50_000):
    data = batch.to_pydict()
    total_rows += batch.num_rows

    for column, values in data.items():
        missing_counts[column] += sum(
            value is None
            or value == ""
            or (column == "added_by_status" and value == "None")
            for value in values
        )

    rawg_ids.update(data["id"])

rawg_missing = pd.DataFrame({
    "missing_count": missing_counts
})

rawg_missing["missing_percent"] = (
    rawg_missing["missing_count"] / total_rows * 100
).round(2)

rawg_missing = rawg_missing.sort_values(
    "missing_percent", ascending=False
)

print("Количество записей:", total_rows)
print("Количество уникальных id:", len(rawg_ids))
print("Количество повторяющихся id:", total_rows - len(rawg_ids))

display(rawg_missing)

Количество записей: 899585
Количество уникальных id: 899585
Количество повторяющихся id: 0


,missing_count,missing_percent
clip,899585,100.00
additions_count,897779,99.80
reddit_count,897701,99.79
tba,896846,99.70
parents_count,896219,99.63
metacritic_platforms,895232,99.52
metacritic,892455,99.21
twitch_count,889731,98.90
reviews_text_count,889180,98.84
game_series_count,888769,98.80


### 2.3.1. Профилирование данных Twitch

Twitch используется для исследования зрительского интереса к видеоиграм и активности стримеров.

Рассматриваются два набора данных: динамика игровых трансляций и просмотров, а также характеристики популярных стримеров.

Для каждого файла проверяются размерность, структура столбцов, типы данных, пропуски и полные дубликаты записей.

In [28]:
twitch_datasets = {
    "Twitch Game Data": RAW / "twitch" / "evolution" / "Twitch_game_data.csv",
    "Twitch Global Data": RAW / "twitch" / "evolution" / "Twitch_global_data.csv",
    "Twitch Streamers": RAW / "twitch" / "streamers" / "datasetV2.csv",
}

twitch_profiles = {}

for name, path in twitch_datasets.items():
    df = pd.read_csv(path, encoding="cp1252")

    twitch_profiles[name] = df

    print(f"\n{'=' * 50}")
    print(name)
    print("Количество записей:", len(df))
    print("Количество столбцов:", len(df.columns))
    print("Полных дубликатов:", df.duplicated().sum())

    print("\nСтолбцы и типы данных:")
    display(df.dtypes.to_frame("dtype"))

    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print("\nПропуски:")
    if missing.empty:
        print("Пропуски отсутствуют.")
    else:
        display(missing.to_frame("missing_count"))


Twitch Game Data
Количество записей: 21000
Количество столбцов: 12
Полных дубликатов: 0

Столбцы и типы данных:


,dtype
Rank,int64
Game,object
Month,int64
Year,int64
Hours_watched,int64
Hours_streamed,int64
Peak_viewers,int64
Peak_channels,int64
Streamers,int64
Avg_viewers,int64



Пропуски:


,missing_count
Game,1



Twitch Global Data
Количество записей: 105
Количество столбцов: 9
Полных дубликатов: 0

Столбцы и типы данных:


,dtype
year,int64
Month,int64
Hours_watched,int64
Avg_viewers,int64
Peak_viewers,int64
Streams,int64
Avg_channels,int64
Games_streamed,int64
Viewer_ratio,float64



Пропуски:
Пропуски отсутствуют.

Twitch Streamers
Количество записей: 999
Количество столбцов: 17
Полных дубликатов: 0

Столбцы и типы данных:


,dtype
RANK,int64
NAME,object
LANGUAGE,object
TYPE,object
MOST_STREAMED_GAME,object
2ND_MOST_STREAMED_GAME,object
AVERAGE_STREAM_DURATION,float64
FOLLOWERS_GAINED_PER_STREAM,int64
AVG_VIEWERS_PER_STREAM,int64
AVG_GAMES_PER_STREAM,float64



Пропуски:


,missing_count
2ND_MOST_STREAMED_GAME,77


### 2.4.1 Профилирование данных YouTube

На данном этапе оценивается качество метаданных видеороликов YouTube. Проверяются структура таблицы, типы данных, пропущенные значения, полные дубликаты и уникальность идентификаторов видео.

Основное внимание уделяется идентификаторам видео, названиям игр, датам публикации и показателям пользовательской активности.

In [29]:
youtube_path = RAW / "youtube" / "youtube_videos.csv"

youtube_df = pd.read_csv(youtube_path)

print("Количество записей:", len(youtube_df))
print("Количество столбцов:", len(youtube_df.columns))
print("Полных дубликатов:", youtube_df.duplicated().sum())

print("\nСтолбцы и типы данных:")
display(youtube_df.dtypes.to_frame("dtype"))

print("\nПропуски:")
youtube_missing = youtube_df.isna().sum()
youtube_missing = youtube_missing[youtube_missing > 0].sort_values(
    ascending=False
)

if youtube_missing.empty:
    print("Пропуски отсутствуют.")
else:
    display(youtube_missing.to_frame("missing_count"))

print("\nУникальность video_id:")
print("Уникальных видео:", youtube_df["video_id"].nunique())
print("Повторяющихся video_id:", youtube_df["video_id"].duplicated().sum())

Количество записей: 299
Количество столбцов: 9
Полных дубликатов: 32

Столбцы и типы данных:


,dtype
game_name,object
video_id,object
title,object
description,object
channel_title,object
published_at,object
views,int64
likes,float64
comments,float64



Пропуски:


,missing_count
description,25
likes,4
comments,1



Уникальность video_id:
Уникальных видео: 267
Повторяющихся video_id: 32


#### 2.4.2 Профилирование дополнительных JSON-источников YouTube

Дополнительно рассматриваются JSON-файлы с результатами поиска и статистикой видеороликов. Проверяется их структура и возможность использования для дополнения основной таблицы метаданных.

In [30]:
youtube_json_files = {
    "Результаты поиска": RAW / "youtube" / "youtube_search_results.json",
    "Статистика видео": RAW / "youtube" / "youtube_video_statistics.json",
}

for name, path in youtube_json_files.items():
    print(f"\n{'=' * 50}")
    print(name)

    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)

    print("Тип корневого объекта:", type(data).__name__)

    if isinstance(data, dict):
        print("Количество ключей:", len(data))
        print("Ключи:", list(data.keys())[:20])

    elif isinstance(data, list):
        print("Количество элементов:", len(data))

        if data:
            print("Тип первого элемента:", type(data[0]).__name__)

            if isinstance(data[0], dict):
                print("Поля первого элемента:", list(data[0].keys()))
                display(pd.DataFrame(data[:3]))
            else:
                display(data[:3])


Результаты поиска
Тип корневого объекта: list
Количество элементов: 299
Тип первого элемента: dict
Поля первого элемента: ['game_name', 'video_id', 'title', 'description', 'channel_title', 'published_at', 'views', 'likes', 'comments']


,game_name,video_id,title,description,channel_title,published_at,views,likes,comments
0,Counter-Strike 2,cXUmsqtlpAw,GRAND FINAL! Spirit vs FURIA - Official Highli...,Check out the match page here for in depth sta...,BLAST Premier CS2 Highlights,2026-11-15T11:25:17Z,361715,4441,464
1,Counter-Strike 2,QVctog5F_jY,Counter Strike 2 Ranked Gameplay 4K (No Commen...,Counter Strike 2 Ranked Gameplay 4K (No Commen...,Hytron,2024-04-24T13:00:32Z,2364155,19912,443
2,Counter-Strike 2,x-m24USPutM,Counter Strike 2 Gameplay 4K (No Commentary),Counter Strike 2 Ranked Gameplay 4K (No Commen...,Hytron,2023-11-30T13:00:49Z,95640,769,16



Статистика видео
Тип корневого объекта: dict
Количество ключей: 267
Ключи: ['NJK-rXgT40E', 'v5ZWGS5aaJU', 'JOClo5aYqHU', '8X2kIfS6fb8', 'UUGCFzUxj94', 'TNygv-tTGvg', 'mYbs3YZgEcs', '-ZQ2Q1c4p-8', 't05rY-_ES4Y', 'G1atkq4C1KU', 'lemSV59wcag', 'hjw-1B2nNtw', '76wIRNsDPPw', 'rk9wHKHAOoA', 'l7B07V4tc3o', '__dvSAqJgTg', 'TWOkT7l0yWQ', 'BI9fgQY0d5I', 'ot7uXNQskhs', 'Qm27HaOkJv4']


In [31]:
stats_path = RAW / "youtube" / "youtube_video_statistics.json"

with open(stats_path, "r", encoding="utf-8") as file:
    video_stats = json.load(file)

video_id, stats = next(iter(video_stats.items()))

print("Количество видео:", len(video_stats))
print("Пример video_id:", video_id)
print("Тип значения:", type(stats).__name__)

if isinstance(stats, dict):
    print("Поля статистики:", list(stats.keys()))
    display(pd.DataFrame([stats]))
else:
    print("Пример значения:", stats)

Количество видео: 267
Пример video_id: NJK-rXgT40E
Тип значения: dict
Поля статистики: ['viewCount', 'likeCount', 'favoriteCount', 'commentCount']


,viewCount,likeCount,favoriteCount,commentCount
0,21251493,501601,0,5864


### 2.5.1. Профилирование текстовых данных Habr

На данном этапе оценивается структура собранных публикаций Habr. Проверяются количество статей, состав атрибутов и наличие текстового содержимого.

Данные рассматриваются как неструктурированный текстовый источник, который в дальнейшем может использоваться для анализа тематик и содержательных особенностей публикаций об игровой индустрии.

In [32]:
habr_path = RAW / "habr" / "articles.json"

with open(habr_path, "r", encoding="utf-8") as file:
    habr_data = json.load(file)

print("Тип корневого объекта:", type(habr_data).__name__)
print("Количество статей:", len(habr_data) if isinstance(habr_data, list) else len(habr_data))

if isinstance(habr_data, list) and habr_data:
    print("\nПоля статьи:")
    print(list(habr_data[0].keys()))

    display(pd.DataFrame(habr_data[:3]).drop(
        columns=["text", "content", "body"], errors="ignore"
    ))

    text_fields = [
        column for column in habr_data[0]
        if column.lower() in ["text", "content", "body", "article_text"]
    ]

    print("\nПоля с текстом:", text_fields)

    if text_fields:
        field = text_fields[0]
        print("Длина текста первой статьи:", len(str(habr_data[0].get(field, ""))))

Тип корневого объекта: list
Количество статей: 2

Поля статьи:
['url', 'title', 'status_code', 'text']


,url,title,status_code
0,https://habr.com/ru/articles/995396/,"Только 12,5% игр в Steam получают пользователь...",200
1,https://habr.com/ru/articles/990554/,От ниши к мейнстриму: с 33 до 75% за 5 лет. Ка...,200



Поля с текстом: ['text']
Длина текста первой статьи: 5720


### 2.6.1. Профилирование иерархических данных Wikidata XML

На данном этапе исследуется структура XML-выгрузки Wikidata, содержащей сведения о видеоиграх. Проверяются корневой элемент, структура вложенности и состав атрибутов записей.

Источник рассматривается как иерархический набор данных, потенциально пригодный для дополнения сведений об играх и их интеграции с другими источниками.

In [33]:
import xml.etree.ElementTree as ET

wikidata_path = RAW / "xml" / "wikidata" / "wikidata_video_games.xml"

def clean_tag(tag):
    return tag.split("}")[-1] if "}" in tag else tag

depth = 0
root_tag = None
record_count = 0
sample_records = []

for event, elem in ET.iterparse(wikidata_path, events=("start", "end")):
    if event == "start":
        depth += 1

        if depth == 1:
            root_tag = clean_tag(elem.tag)

    else:
        if depth == 2:
            record_count += 1

            if len(sample_records) < 3:
                sample_records.append({
                    "tag": clean_tag(elem.tag),
                    "attributes": elem.attrib,
                    "children": [
                        clean_tag(child.tag) for child in elem
                    ],
                    "sample_values": {
                        clean_tag(child.tag): (child.text or "").strip()[:150]
                        for child in elem
                        if child.text and child.text.strip()
                    }
                })

            elem.clear()

        depth -= 1

print("Корневой элемент:", root_tag)
print("Количество записей первого уровня:", record_count)

for i, record in enumerate(sample_records, 1):
    print(f"\nЗапись {i}:")
    print("Тег:", record["tag"])
    print("Атрибуты:", record["attributes"])
    print("Вложенные элементы:", record["children"])
    print("Примеры значений:", record["sample_values"])

Корневой элемент: sparql
Количество записей первого уровня: 2

Запись 1:
Тег: head
Атрибуты: {}
Вложенные элементы: ['variable', 'variable', 'variable', 'variable', 'variable', 'variable', 'variable']
Примеры значений: {}

Запись 2:
Тег: results
Атрибуты: {}
Вложенные элементы: ['result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 'result', 